# Fake News Detection In Arabic Newspapers

### imports

In [ ]:
import os
import pandas as pd
import itertools
import numpy as np
import string
import re
import matplotlib.pyplot as plt

import pickle

from sklearn import preprocessing
import matplotlib
from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer


# Keras
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Dense, Flatten, LSTM, Conv1D, MaxPooling1D, Dropout, Activation,SpatialDropout1D
from keras.layers.embeddings import Embedding


# Stemming

from nltk.stem import SnowballStemmer


## Normalisation

### remove diacritics

In [ ]:
def removeDiacretics(news_list):

    arabic_diacritics = re.compile("""
                             ?    | # Tashdid
                             ?    | # Fatha
                             ?    | # Tanwin Fath
                             ?    | # Damma
                             ?    | # Tanwin Damm
                             ?    | # Kasra
                             ?    | # Tanwin Kasr
                             ?    | # Sukun
                             ?     # Tatwil/Kashida
                         """, re.VERBOSE)
    temp_list = list()
    for news in news_list:
        text = re.sub(arabic_diacritics, '', news)
        temp_list.append(text)

    return temp_list


### get top n words

In [ ]:
def get_top_n_words(corpus, n=None):
   
    vec = CountVectorizer().fit(corpus)
    bag_of_words = vec.transform(corpus)
    sum_words = bag_of_words.sum(axis=0)
    words_freq = [(word, sum_words[0, idx]) for word, idx in vec.vocabulary_.items()]
    words_freq =sorted(words_freq, key=lambda x: x[1], reverse=True)
    return words_freq[:n]


### normalise arabic

In [ ]:
def normalize_arabic(text):
    text = re.sub("[????]", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    return text

### other normalisations

In [ ]:


def remove_diacritics(text):
    text = re.sub(arabic_diacritics, '', text)
    return text

def remove_punctuations(text):
    translator = str.maketrans(' ', ' ', punctuations_list)
    return text.translate(translator)

def remove_repeating_char(text):
    return re.sub(r'(.)\1+', r'\1', text)

### remove ponctuations

In [ ]:

arabic_punctuations = '''`???<>_()*&^%][??/:"?.,'{}~?+|"!?????'''
english_punctuations = string.punctuation
punctuations_list = arabic_punctuations + english_punctuations


### stopwords

In [ ]:

stop_words = set(stopwords.words('arabic'))


### normalise arabic

In [ ]:
def normalizeArabic(text):
    text = re.sub("[?????]", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    text = re.sub("?", "?", text)
    return(text)


In [ ]:

ar_stemmer = SnowballStemmer("arabic")

trainDF1 = pd.read_csv("dataset.csv", delimiter=",", encoding='utf-8')
trainDF2 = pd.read_csv("dataset2.csv", delimiter=",", encoding='utf-8')

trainDF = pd.concat([trainDF1,trainDF2])

trainDF.text[0]

pd.DataFrame(trainDF.text[0]).to_csv("1.csv",index=False)



trainDF['label'].value_counts().plot(kind='bar')




### Apply normalise to all dataset

In [ ]:
def normalizeDF(df):

    for index, row in df.iterrows():

        text = df.iloc[index].text
        
        if type(text) == str:
            
            text = remove_punctuations(text).rstrip()

            text = removeDiacretics([text])

            word_tokens = word_tokenize(text[0])

            filtered_sentence = [w for w in word_tokens if not w in stop_words]

            stemmed_words = [ar_stemmer.stem(word) for word in filtered_sentence]

            stemmed_sentence = ' '.join(stemmed_words)

            df.iloc[index].text = stemmed_sentence

    return df

In [ ]:

filtered_df = normalizeDF(trainDF)

myinp = filtered_df.text[0]

pd.DataFrame(filtered_df.text[0]).to_csv("2.csv",index=False)

filtered_df.to_csv('to_train.csv', sep=',')

print(filtered_df.head())

filtered_df.to_csv('to_train.csv', sep=',')

print("filtered")



#### split dataset 

In [ ]:
train_x, test_x, train_y, test_y = train_test_split(trainDF['text'].values, trainDF['label'].values,test_size=0.2)

print(train_x.shape, train_y.shape)
print(test_x.shape, test_y.shape)

### tfidfVectorizer

In [ ]:

max_fatures = 500

# word level tf-idf
tfidf_vect = TfidfVectorizer(analyzer='word',max_features=5000)
tfidf_vect.fit(trainDF['text'].values.astype('U'))


print(type(train_x))
print(train_x.shape)
xtrain_tfidf = tfidf_vect.transform(train_x.astype('U')).toarray()

xvalid_tfidf = tfidf_vect.transform(test_x.astype('U')).toarray()

pickle.dump(tfidf_vect, open("vectorizer.pickle", "wb"))

mdl2 = pickle.load(open("vectorizer.pickle","rb"))

myinp = '''


???? 18 ??? ??? ? 3 ???? ?? ???? ????? ???? ??? ????? ???? ???? ???? ???? ???? ??? ???? ??? ??? ???? ??? ??? ??? ?? ???? ???? ? ?????? ????? ???? ????? ???? ??? 18 ??? ??? ?3 ???? ???? ???? ????? ??? ???? 4 ???? ?9 ??? ??? ??? ??? ???????? ???? ???? ??? ???? ??? ??? ??? ???? ??? ????? ??? ???? ???? ???? ??? ???? ???? ???? ??? ???? ???? ?????? ???? ???? ????? ??? ???? ???? ??? ???? ???? ????? ???? ??? ????? ???? ????? ???? ??? ???? ???? ???? ????? ??? ??? ????? ???? ???? ???? ???? ?????? ???? ???? ??? ??? ???? ??? ??? ?? ???? ????? ???? ??? ???? ??? ????? ??? ???? ??? ??? ???? ??? ??? ??? ????? ????? ?????? ????? ??? ?? ???? ???? ??? ???? ??? ??? ??? ???? ??????? ???? ??? ??? ????? ???? ???? ??? ???? ??????????? ??? ?? ???? 19 ???? ??? ???? ???? ??? ??? ??? ????? ????? ??? ??? ??? ??? ??? ??????? ????? ??? ???????????? ??? ????? ??? ????? ???? ????? ???? ???? ????? ???? ???? ???? ??? ??? ????? ????? ??? ???? ??? ???? ???? ???? 3 ????? ??? ???? ?? ??? ?? ????? ??? ???? ???? ???? ????? ??? ?? ??? ?? ?? ???? ?????? ??? ????? ???? ??? ????? ????? ??? ???? ???? ????? ????? ???? ???? ??? ???? ???? ??? ????? ????? ????? ??? ???? ????? ???? ????? ???? ??? ?????? ??? ??? ???? ???? ????? ????? ??????? ?? ??? ???? ?? ???? ????? ??? ??? ??? ???? ???? ?? ??? ???? ????


'''

print(train_x[0])



myinp0 = tfidf_vect.transform(np.array([train_x[0]]).astype('U')).toarray()
myinpvect = mdl2.transform(np.array([myinp]).astype('U')).toarray()



pd.DataFrame(myinp0).to_csv("k1.csv")
pd.DataFrame(myinpvect).to_csv("k2.csv")


pd.DataFrame(xtrain_tfidf[0]).to_csv("1.csv")
# save model


print(xtrain_tfidf.shape)
print(xvalid_tfidf.shape)

# load model 

### CountVectorizer

In [ ]:

count_vec = CountVectorizer(analyzer='word', max_features=5000)
count_train = count_vec.fit(trainDF['text'].values.astype('U'))
bag_of_words_train = count_vec.transform(train_x.astype('U')).toarray()
bag_of_words_test = count_vec.transform(test_x.astype('U')).toarray()



print(bag_of_words_train.shape)
print(bag_of_words_test.shape)


## Logistic Regression +TfIdfVectorizer

In [ ]:

# Fitting the Logistic Regression into the Training set



from sklearn.metrics import accuracy_score




from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state = 0)
classifier.fit(xtrain_tfidf, train_y)


# save resulting model


pickle.dump(classifier, open("logistic_reg.pickle", "wb"))

# Predicting the test set results

tested_y = classifier.predict(xvalid_tfidf)

print(tested_y.shape)
print(train_y.shape)

print('Validation Accuracy : ' + str(accuracy_score(test_y, tested_y)))




# Making the Confusion Matrix

print(tested_y.shape)
print(test_y.shape)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(tested_y, test_y)

def plot_confusion_matrix(cm, classes=['not fake','fake'],
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm,
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, normalize=True,
                      title='Normalized confusion matrix')

plt.show()



## Logistic Regression + CountVectorizer

In [ ]:

classifier = LogisticRegression(random_state = 0)
classifier.fit(bag_of_words_train, train_y)

# Predicting the test set results

tested_y = classifier.predict(bag_of_words_test)

print(tested_y.shape)
print(test_y.shape)

print('Validation Accuracy : ' + str(accuracy_score(test_y, tested_y)))

# Making the Confusion Matrix

print(tested_y.shape)
print(test_y.shape)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(tested_y, test_y)

def plot_confusion_matrix(cm, classes=['not fake','fake'],
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm,
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, normalize=True,
                      title='Normalized confusion matrix')

plt.show()


## Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model1 = MultinomialNB().fit( xtrain_tfidf, train_y)

# save model

pickle.dump(model1, open("nayve_bayes.pickle", "wb"))


tested_y = model1.predict(xvalid_tfidf)


print(tested_y.shape)
print(train_y.shape)

print('Validation Accuracy : ' + str(accuracy_score(test_y, tested_y)))


print(tested_y.shape)
print(test_y.shape)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(tested_y, test_y)

def plot_confusion_matrix(cm, classes=['not fake','fake'],
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm,
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, normalize=True,
                      title='Normalized confusion matrix')

plt.show()




## Random Forest Classifier

In [ ]:
#Import Random Forest Model
from sklearn.ensemble import RandomForestClassifier

#Create a Gaussian Classifier
clf=RandomForestClassifier(n_estimators=100)



#Train the model using the training sets y_pred=clf.predict(X_test)
clf.fit(bag_of_words_train, train_y)

# save model

pickle.dump(clf, open("random_forest.pickle", "wb"))



tested_y=clf.predict(bag_of_words_test)


print('Validation Accuracy : ' + str(accuracy_score(test_y, tested_y)))


print(tested_y.shape)
print(test_y.shape)
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(tested_y, test_y)

def plot_confusion_matrix(cm, classes=['not fake','fake'],
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()


# Compute confusion matrix
np.set_printoptions(precision=2)

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm,
                      title='Confusion matrix, without normalization')

# Plot normalized confusion matrix
plt.figure()
plot_confusion_matrix(cm, normalize=True,
                      title='Normalized confusion matrix')

plt.show()



In [ ]:
news_input = '''

???? 18 ??? ??? ? 3 ???? ?? ???? ????? ???? ??? ????? ???? ???? ???? ???? ???? ??? ???? ??? ??? ???? ??? ??? ??? ?? ???? ???? ? ?????? ????? ???? ????? ???? ??? 18 ??? ??? ?3 ???? ???? ???? ????? ??? ???? 4 ???? ?9 ??? ??? ??? ??? ???????? ???? ???? ??? ???? ??? ??? ??? ???? ??? ????? ??? ???? ???? ???? ??? ???? ???? ???? ??? ???? ???? ?????? ???? ???? ????? ??? ???? ???? ??? ???? ???? ????? ???? ??? ????? ???? ????? ???? ??? ???? ???? ???? ????? ??? ??? ????? ???? ???? ???? ???? ?????? ???? ???? ??? ??? ???? ??? ??? ?? ???? ????? ???? ??? ???? ??? ????? ??? ???? ??? ??? ???? ??? ??? ??? ????? ????? ?????? ????? ??? ?? ???? ???? ??? ???? ??? ??? ??? ???? ??????? ???? ??? ??? ????? ???? ???? ??? ???? ??????????? ??? ?? ???? 19 ???? ??? ???? ???? ??? ??? ??? ????? ????? ??? ??? ??? ??? ??? ??????? ????? ??? ???????????? ??? ????? ??? ????? ???? ????? ???? ???? ????? ???? ???? ???? ??? ??? ????? ????? ??? ???? ??? ???? ???? ???? 3 ????? ??? ???? ?? ??? ?? ????? ??? ???? ???? ???? ????? ??? ?? ??? ?? ?? ???? ?????? ??? ????? ???? ??? ????? ????? ??? ???? ???? ????? ????? ???? ???? ??? ???? ???? ??? ????? ????? ????? ??? ???? ????? ???? ????? ???? ??? ?????? ??? ??? ???? ???? ????? ????? ??????? ?? ??? ???? ?? ???? ????? ??? ??? ??? ???? ???? ?? ??? ???? ????
'''
print(news_input)

empty_list = list()
empty_list.append(news_input)
    
print(empty_list)

filtered_df = normalizeDF(pd.DataFrame(empty_list,columns=['text']))

print("here")

print(filtered_df)



    
model1_name = "logistic_reg.pickle"
model2_name = "nayve_bayes.pickle"
model3_name = "random_forest.pickle"

print('on est la mon frere')    
    
    

loaded_model1 = pickle.load(open(model1_name, 'rb'))
loaded_model2 = pickle.load(open(model2_name, 'rb'))
loaded_model3 = pickle.load(open(model3_name, 'rb'))
    
tfidfvect = pickle.load(open("vectorizer.pickle","rb"))


news_vector = tfidfvect.transform(filtered_df.values.astype('U')[0]).toarray()

pd.DataFrame(filtered_df.values).to_csv("k3.csv")

pd.DataFrame(news_vector).to_csv("K4.csv")

print(loaded_model1.predict(news_vector))
print(loaded_model2.predict(news_vector))
print(loaded_model3.predict(news_vector))
    
print('loaded_model')
    



In [ ]:
import keras

Y = pd.get_dummies(train_y)
Y_t = pd.get_dummies(test_y)


model = Sequential()

model.add(Conv1D(128, 5,padding='same',
                 input_shape=(5000,1)))
model.add(Activation('relu'))
model.add(Conv1D(128, 5,padding='same'))
model.add(Activation('relu'))
model.add(Dropout(0.1))
model.add(MaxPooling1D(pool_size=(8)))
model.add(Conv1D(128, 5,padding='same',))
model.add(Activation('relu'))
model.add(Conv1D(128, 5,padding='same',))
model.add(Activation('relu'))

model.add(Flatten())
model.add(Dense(2))
model.add(Activation('softmax'))
opt = keras.optimizers.rmsprop(lr=0.00001, decay=1e-6)
model.summary()
model.compile(loss='categorical_crossentropy', optimizer=opt,metrics=['accuracy'])
cnnhistory=model.fit(bag_of_words_train.reshape((7398,5000,1)), Y, batch_size=32, epochs=10, validation_data=(bag_of_words_test, y_t))


In [ ]:
import keras

Y = pd.get_dummies(train_y)
Y_t = pd.get_dummies(test_y)


model = Sequential()

model.add(Conv1D(128, 5,padding='same',
                 input_shape=(5000,1)))
model.add(Activation('relu'))
model.add(Conv1D(128, 5,padding='same'))
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(MaxPooling1D(pool_size=(8)))
model.add(LSTM(100))
model.add(Flatten())
model.add(Dense(2))
model.add(Activation('softmax'))
opt = keras.optimizers.rmsprop(lr=0.00001, decay=1e-6)
model.summary()
model.compile(loss='categorical_crossentropy', optimizer=opt,metrics=['accuracy'])
cnnhistory=model.fit(bag_of_words_train.reshape((7398,5000,1)), Y, batch_size=32, epochs=10, validation_data=(bag_of_words_test, y_t))


## Long Short Term Memory